## Case When formatting test

### Goals

Answer question from `orbital` team: 

> Does "unnesting" queries improve performance for trees? If only sometimes, where/when?

Research design:

- Simulate SQL code for medium-size random forest:
  + Max Depth: 4 (2**4 terminal nodes)
  + Trees: 100
- Test three different varieties for performance:
  + Original nested CASE WHEN
  + "Linearized" CASE WHEN (pulling all conditions to top level)
  + "Smart" Linearized -- same as above but with refundant conditions removed
- Test on two different data distributions:
  + Uniformly distributed (all nodes have same chance of terminating)
  + Skewed distribution (to simulate if we re-rodered query to take advantage of CASE WHEN early term)


### Set Up -- Tree Code

In [1]:
import duckdb
import polars as pl
import polars.selectors as cs
import sqlglot
import pandas as pd
import numpy as np

In [2]:
# nested case when version
d = []
for i in np.arange(8):
    d += [ f"case when d{i+1} > 0.5 then {i+1} else 0 end"]

c = []
for i in np.arange(4):
    sql_temp = f"case when c{i+1} > 0.5 then {d[2*i]} else {d[2*i+1]} end"
    c += [sql_temp]

b = []
for i in np.arange(2):
    sql_temp = f"case when b{i+1} > 0.5 then {c[2*i]} else {c[2*i+1]} end"
    b += [sql_temp]

sql_nest = f"case when a > 0.5 then {b[0]} else {b[1]} end"

print(
    sqlglot.transpile(sql_nest, write="duckdb", identify=True, pretty=True)[0]
)

CASE
  WHEN "a" > 0.5
  THEN CASE
    WHEN "b1" > 0.5
    THEN CASE
      WHEN "c1" > 0.5
      THEN CASE WHEN "d1" > 0.5 THEN 1 ELSE 0 END
      ELSE CASE WHEN "d2" > 0.5 THEN 2 ELSE 0 END
    END
    ELSE CASE
      WHEN "c2" > 0.5
      THEN CASE WHEN "d3" > 0.5 THEN 3 ELSE 0 END
      ELSE CASE WHEN "d4" > 0.5 THEN 4 ELSE 0 END
    END
  END
  ELSE CASE
    WHEN "b2" > 0.5
    THEN CASE
      WHEN "c3" > 0.5
      THEN CASE WHEN "d5" > 0.5 THEN 5 ELSE 0 END
      ELSE CASE WHEN "d6" > 0.5 THEN 6 ELSE 0 END
    END
    ELSE CASE
      WHEN "c4" > 0.5
      THEN CASE WHEN "d7" > 0.5 THEN 7 ELSE 0 END
      ELSE CASE WHEN "d8" > 0.5 THEN 8 ELSE 0 END
    END
  END
END
CASE
  WHEN "a" > 0.5
  THEN CASE
    WHEN "b1" > 0.5
    THEN CASE
      WHEN "c1" > 0.5
      THEN CASE WHEN "d1" > 0.5 THEN 1 ELSE 0 END
      ELSE CASE WHEN "d2" > 0.5 THEN 2 ELSE 0 END
    END
    ELSE CASE
      WHEN "c2" > 0.5
      THEN CASE WHEN "d3" > 0.5 THEN 3 ELSE 0 END
      ELSE CASE WHEN "d4" > 0.5 THEN 4

In [3]:
# linearize case when version 
# using strict versus equalities versus inclusion / exclusion just so my code lines up nice =) 
# it's a timing exercise so it really doesn't matter
sql_line = '''
case
-- a > 0.5
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 < 0.5 then 0 
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5 and c1 < 0.5 and d2 < 0.5 then 0
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and b1 < 0.5 and c2 > 0.5 and d3 < 0.5 then 0 
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 > 0.5 then 4
when a > 0.5 and b1 < 0.5 and c2 < 0.5 and d4 < 0.5 then 0
-- a <= 0.5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when a < 0.5 and b2 > 0.5 and c3 > 0.5 and d5 < 0.5 then 0 
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 > 0.5 then 6
when a < 0.5 and b2 > 0.5 and c3 < 0.5 and d6 < 0.5 then 0
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 > 0.5 then 7
when a < 0.5 and b2 < 0.5 and c4 > 0.5 and d7 < 0.5 then 0 
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 > 0.5 then 8
when a < 0.5 and b2 < 0.5 and c4 < 0.5 and d8 < 0.5 then 0
else null end
'''

print(
    sqlglot.transpile(sql_line, write="duckdb", identify=True, pretty=True)[0]
)

CASE
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" > 0.5
  THEN 1
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" < 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" < 0.5 AND "d2" > 0.5
  THEN 2
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" < 0.5 AND "d2" < 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" > 0.5 AND "d3" > 0.5
  THEN 3
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" > 0.5 AND "d3" < 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" < 0.5 AND "d4" > 0.5
  THEN 4
  WHEN "a" > 0.5 AND "b1" < 0.5 AND "c2" < 0.5 AND "d4" < 0.5
  THEN 0
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" > 0.5 AND "d5" > 0.5
  THEN 5
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" > 0.5 AND "d5" < 0.5
  THEN 0
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" < 0.5 AND "d6" > 0.5
  THEN 6
  WHEN "a" < 0.5 AND "b2" > 0.5 AND "c3" < 0.5 AND "d6" < 0.5
  THEN 0
  WHEN "a" < 0.5 AND "b2" < 0.5 AND "c4" > 0.5 AND "d7" > 0.5
  THEN 7
  WHEN "a" < 0.5 AND "b2" < 0.5 AND "c4" > 0.5 AND "d7" < 0.5
  THEN 0
 

In [4]:
# linearized case when version -- pruning redundant conditions
sql_slim = '''
case
-- a > 0.5
when a > 0.5 and b1 > 0.5 and c1 > 0.5 and d1 > 0.5 then 1
when a > 0.5 and b1 > 0.5 and c1 > 0.5              then 0 
when a > 0.5 and b1 > 0.5              and d2 > 0.5 then 2
when a > 0.5 and b1 > 0.5                           then 0
when a > 0.5 and              c2 > 0.5 and d3 > 0.5 then 3
when a > 0.5 and              c2 > 0.5              then 0 
when a > 0.5 and                           d4 > 0.5 then 4
when a > 0.5                                        then 0
-- a <= 0.5
when             b2 > 0.5 and c3 > 0.5 and d5 > 0.5 then 5
when             b2 > 0.5 and c3 > 0.5              then 0 
when             b2 > 0.5              and d6 > 0.5 then 6
when             b2 > 0.5                           then 0
when                          c4 > 0.5 and d7 > 0.5 then 7
when                          c4 > 0.5              then 0 
when                                       d8 > 0.5 then 8
when a < 0.5                                        then 0
else null end
'''

print(
    sqlglot.transpile(sql_slim, write="duckdb", identify=True, pretty=True)[0]
)

CASE
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" > 0.5
  THEN 1
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "d2" > 0.5
  THEN 2
  WHEN "a" > 0.5 AND "b1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "c2" > 0.5 AND "d3" > 0.5
  THEN 3
  WHEN "a" > 0.5 AND "c2" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "d4" > 0.5
  THEN 4
  WHEN "a" > 0.5
  THEN 0
  WHEN "b2" > 0.5 AND "c3" > 0.5 AND "d5" > 0.5
  THEN 5
  WHEN "b2" > 0.5 AND "c3" > 0.5
  THEN 0
  WHEN "b2" > 0.5 AND "d6" > 0.5
  THEN 6
  WHEN "b2" > 0.5
  THEN 0
  WHEN "c4" > 0.5 AND "d7" > 0.5
  THEN 7
  WHEN "c4" > 0.5
  THEN 0
  WHEN "d8" > 0.5
  THEN 8
  WHEN "a" < 0.5
  THEN 0
  ELSE NULL
END
CASE
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5 AND "d1" > 0.5
  THEN 1
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "c1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "b1" > 0.5 AND "d2" > 0.5
  THEN 2
  WHEN "a" > 0.5 AND "b1" > 0.5
  THEN 0
  WHEN "a" > 0.5 AND "c2" > 0.5 AND "d3" > 0.5
  THEN 3
  WHEN "a" > 0.5

### Set Up -- Data

In [5]:
# set up random data matrix
n = 1000000
p = 15
df = pl.DataFrame( np.random.rand(n,p) )
df.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']
df.glimpse()

Rows: 1000000
Columns: 15
$ a  <f64> 0.14742862407897683, 0.24509657297465526, 0.9157016094920555, 0.46049170704702747, 0.16646005656379437, 0.4699246791623134, 0.09584973483097892, 0.17292273422153204, 0.7094978449182209, 0.9598546374895349
$ b1 <f64> 0.3858745827643171, 0.43421161692908283, 0.7202045582793047, 0.0846856930359059, 0.6886670916128049, 0.49167612297020047, 0.4155659640761824, 0.157531920691674, 0.3427860858381807, 0.7957118480893528
$ b2 <f64> 0.3164442816774753, 0.6001987158026171, 0.4112645835076493, 0.40070428002129854, 0.7151980823009808, 0.016709009126246843, 0.4166904671360554, 0.9857036022423527, 0.49329682662496055, 0.5447655925714029
$ c1 <f64> 0.6060068316712116, 0.9217608598774895, 0.33719086675413257, 0.522525256002867, 0.5260573938393311, 0.05460309179320233, 0.24268677510969883, 0.7169707516332251, 0.0349483301363418, 0.09706311781402266
$ c2 <f64> 0.9262130447105582, 0.9293303278684152, 0.6462818995845517, 0.09518633400817789, 0.30238470887198365, 0.00656

In [6]:
# ensure different candidates have same logic
sql_compare = f"""
select
{sql_nest} as out_nest,
{sql_line} as out_line,
{sql_slim} as out_slim,
*
from df
"""
df_out = duckdb.sql(sql_compare).pl()
df_out.filter( 
    (pl.col('out_line') != pl.col('out_slim')) |
    (pl.col('out_nest') != pl.col('out_slim'))
)

out_nest,out_line,out_slim,a,b1,b2,c1,c2,c3,c4,d1,d2,d3,d4,d5,d6,d7,d8
i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [7]:
# base case output frequency
df_out.group_by('out_nest').len()

out_nest,len
i32,u32
0,500519
6,62691
7,62575
8,62148
3,62467
5,62592
4,62285
2,62233
1,62490


In [8]:
# create different skews
df_skew = (
df_out
.with_columns(threshhold = pl.when(pl.col('out_nest') > 0).then(10 - pl.col('out_nest')).otherwise(5))
.filter( pl.col('out_nest').cum_count().over('out_nest') <= pl.col('threshhold')*7000 )
)

print(df_skew.shape[0])

(
df_skew
.group_by('out_nest')
.len()
.sort('out_nest')
.with_columns( p = pl.col('len') / pl.col('len').sum() )
)

342490
342490


out_nest,len,p
i32,u32,f64
0,35000,0.102193
1,62490,0.182458
2,56000,0.163508
3,49000,0.14307
4,42000,0.122631
5,35000,0.102193
6,28000,0.081754
7,21000,0.061316
8,14000,0.040877


In [9]:
# final prep - standardizing data size
df_skew = pl.concat([df_skew]*4).drop( cs.starts_with('out_') )
df_unif = pl.DataFrame( np.random.rand( df_skew.shape[0],p) )
df_unif.columns = ['a','b1','b2','c1','c2','c3','c4','d1','d2','d3','d4','d5','d6','d7','d8']

In [10]:
### Timing

con = duckdb.connect()
con.sql("SET enable_object_cache = false;")

#### Base Case

In this case all nodes are equally likely

In [11]:
qry_nest = f"select {'+'.join([sql_nest]*100)} as pred from df_unif"
qry_line = f"select {'+'.join([sql_line]*100)} as pred from df_unif"
qry_slim = f"select {'+'.join([sql_slim]*100)} as pred from df_unif"

In [12]:
%%timeit -n 1 -r 50

con.sql(qry_nest).execute()

673 ms ± 31.8 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
673 ms ± 31.8 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [13]:
%%timeit -n 1 -r 50

con.sql(qry_line).execute()

2.5 s ± 93.1 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
2.5 s ± 93.1 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [14]:
%%timeit -n 1 -r 50

con.sql(qry_slim).execute()

1.43 s ± 48.5 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.43 s ± 48.5 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


#### Skewed Case

In this case, nodes are skewed, so we can see benefit from the linearized version ordering by node size 

In [15]:
qry_nest = f"select {'+'.join([sql_nest]*100)} as pred from df_skew"
qry_line = f"select {'+'.join([sql_line]*100)} as pred from df_skew"
qry_slim = f"select {'+'.join([sql_slim]*100)} as pred from df_skew"

In [16]:
%%timeit -n 1 -r 50

con.sql(qry_nest).execute()

673 ms ± 39.7 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
673 ms ± 39.7 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [19]:
%%timeit -n 1 -r 50

con.sql(qry_line).execute()

2.49 s ± 132 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
2.49 s ± 132 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)


In [18]:
%%timeit -n 1 -r 50

con.sql(qry_slim).execute()

1.45 s ± 57.3 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
1.45 s ± 57.3 ms per loop (mean ± std. dev. of 50 runs, 1 loop each)
